In [ ]:
import pandas as pd
from pathlib import Path
import os
import openpyxl
import re
import json
import pandas as pd


In [ ]:
def find_project_root(start_path, marker="data"):
    path = Path(start_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_SAMPLES =  PROJECT_ROOT / "data" / "samples"
BENCHMARKS_FINAL = PROJECT_ROOT / "benchmarks" / "final"

In [ ]:
cols = ["transcript_id", "question_order", "output"]

llama_8b = pd.read_csv(
    BENCHMARKS_FINAL / "llama-3.1-8b-instruct_n60_maxtok2500_20260414_122401.csv",
    usecols=cols,
    engine="pyarrow"
)

qwen_4b = pd.read_csv(
    BENCHMARKS_FINAL / 'qwen_qwen3-4b-2507_n60_maxtok2000_20260414_122222.csv',
    usecols=cols,
    engine="pyarrow"
)

qwen_7b = pd.read_csv(
    BENCHMARKS_FINAL / 'qwen2.5-vl-7b-instruct_n60_maxtok2000_20260414_123956.csv',
    usecols=cols,
    engine="pyarrow"
)

qwen_8b = pd.read_csv(
    BENCHMARKS_FINAL / "qwen3-vl-8b-instruct_n60_maxtok2500_20260414_122743.csv",
    usecols=cols,
    engine="pyarrow"
)

gpt_5_5b = pd.read_csv(
    BENCHMARKS_FINAL / "gpt5.5_v2.csv",
    usecols=cols,
    
)

In [ ]:
gpt_5_5b 

In [ ]:
for i in [qwen_7b, qwen_4b, qwen_8b, llama_8b, gpt_5_5b]:
    i["question_order"] = i["question_order"].astype(int)
    print(i.columns)
    print(len(i))   

In [ ]:
qwen_7b = qwen_7b.rename(columns={"output": "outputqwen2.5-7b"})
qwen_4b = qwen_4b.rename(columns={"output": "outputqwen3-4b"})
qwen_8b = qwen_8b.rename(columns={"output": "outputqwen3-8b"})
llama_8b = llama_8b.rename(columns={"output": "outputllama-3.1-8b"})
gpt_5_5b = gpt_5_5b.rename(columns={"output": "outputgpt5.5-5b"})

In [ ]:
for name, df in [
    ("qwen_7b", qwen_7b),
    ("qwen_4b", qwen_4b),
    ("qwen_8b", qwen_8b),
    ("llama_8b", llama_8b),
    ("gpt_5_5b", gpt_5_5b)
]:
    dupes = df.duplicated(subset=["transcript_id", "question_order"]).sum()
    print(f"{name}: {dupes} duplicates")

In [ ]:
merge = (
    qwen_7b
    .merge(qwen_4b, on=["transcript_id", "question_order"])
    .merge(qwen_8b, on=["transcript_id", "question_order"])
    .merge(llama_8b,on=["transcript_id", "question_order"])
    .merge(gpt_5_5b,on=["transcript_id", "question_order"]))

del qwen_7b, qwen_4b, qwen_8b, llama_8b, gpt_5_5b

In [ ]:
models = [
    ("outputqwen2.5-7b", "qwen_7b"),
    ("outputqwen3-4b", "qwen_4b"),
    ("outputqwen3-8b", "qwen_8b"),
    ("outputllama-3.1-8b", "llama_8b"),
    ("outputgpt5.5-5b", "gpt_5_5b")
    
]

In [ ]:
def extract_fields(row, col, source):
    text = row[col]

    if pd.isna(text):
        return None

    try:
        x = json.loads(text)

        return {
            "transcript_id": row["transcript_id"],
            "source": source,
            "fli": x.get("forward_looking_intensity"),
            "spec": x.get("specificity"),
            "sub": x.get("economic_substance"),
            "tone": x.get("tone"),
            "cert": x.get("certainty"),
            "main_focus": x.get("context_summary", {}).get("main_focus"),
            "secondary_focus": x.get("context_summary", {}).get("secondary_focus"),
            "managerial_horizon": x.get("context_summary", {}).get("managerial_horizon"),
            "overall_outlook": x.get("context_summary", {}).get("overall_outlook"),
            "question_order": row["question_order"] ,
        }

    except:
        return None

In [ ]:
merge.columns

In [ ]:
import time

all_results = []

for col, name in models:
    start = time.time()

    temp = merge.apply(lambda row: extract_fields(row, col, name), axis=1)
    temp = temp.dropna().tolist()

    all_results.extend(temp)

    end = time.time()

    print(f"{name} | rows: {len(temp)} | time: {end - start:.2f}s")

    del temp

In [ ]:
df_final = pd.DataFrame(all_results)

In [ ]:
df_final.columns

In [ ]:
llm_long = df_final.melt(
    id_vars=["transcript_id", "question_order", "source"],
    value_vars=["fli", "spec", "sub", "tone", "cert", "main_focus", "secondary_focus", "managerial_horizon", "overall_outlook"],
    var_name="Measure",
    value_name="Score"
)

In [ ]:
own = pd.read_excel(BENCHMARKS_FINAL / "output_own.xlsx")



In [ ]:
own.columns

In [ ]:
rename_map = {
    "fli": "FLI",
    "spec": "Specificity",
    "sub": "EC Sub",
    "tone": "Tone",
    "cert": "Certainty",
    "main_focus": "Main Foc",
    "secondary_focus": "Second Foc",
    "managerial_horizon": "Horizon",
    "overall_outlook": "Outlook"
}
# Apply ONLY to raw names, keep existing correct ones
llm_long["Measure"] = llm_long["Measure"].replace(rename_map)

In [ ]:
numeric_measures = ["FLI", "Specificity", "EC Sub", "Tone", "Certainty"]

df_num = llm_long[llm_long["Measure"].isin(numeric_measures)].copy()

df_num["Score"] = pd.to_numeric(df_num["Score"], errors="coerce")

pivot = df_num.pivot_table(
    index=["transcript_id", "Measure"],
    columns="source",
    values="Score"
)

pivot = pivot.dropna()

pivot.corr()

In [ ]:
print(llm_long["Measure"].value_counts())

In [ ]:
llm_num = llm_long[llm_long["Measure"].isin(numeric_measures)].copy()

llm_num["Score"] = pd.to_numeric(llm_num["Score"], errors="coerce")

llm_num = llm_num.rename(columns={"Score": "Score_llm"})

In [ ]:
own_num = own[own["Measure"].isin(numeric_measures)].copy()

own_num["Score"] = pd.to_numeric(own_num["Score"], errors="coerce")

own_num = own_num.rename(columns={
    "Score": "Score_manual",
    "Source": "source_manual"
})



In [ ]:
merged = llm_num.merge(
    own_num,
    on=["transcript_id", "question_order", "Measure"]
)

In [ ]:
merged.columns

In [ ]:
merged[["Score_llm", "Score_manual"]].corr()

In [ ]:
pivot

In [ ]:
corrs_model = (
    merged
    .groupby(["source", "Measure"], group_keys=False)
    .apply(lambda x: x["Score_llm"].corr(x["Score_manual"]))
    .unstack()
)
print(corrs_model)

In [ ]:
gpt_col = "gpt_5_5b"

gpt_corrs_by_measure = (
    pivot
    .groupby(level="Measure")
    .apply(lambda x: x.corr()[gpt_col].drop(gpt_col))
    .unstack()
)

print(gpt_corrs_by_measure.round(3))